# 22. Vlákna, Paralerní programování, Asynchronní metody, Concurrent design patterns

### Paralelismus vs. Konkurence
* Paralelně znamená, že více procesů běží skutečně ve stejný okamžik na různých hardwarových zdrojích
* V Pythonu knihovna `multiprocessing`
* Souběžnost představuje stav, kdy se různé algoritmy rychle střídají na jednom procesorovém jádru
* Vytváří se tak pouhá iluze paralelního běhu
* V Pythonu knihovna `threading`

### Vlákna a GIL v Pythonu
* `threading` slouží k vytváření vláken, která běží souběžně
* Důvodem je zabudovaný mechanismus zvaný GIL (Global Interpreter Lock)
* Ten garantuje, že na procesoru běží v jeden okamžik pouze jediné vlákno a ostatní se na něm musí střídat
* Oproti procesům, které paměť nesdílejí (vlastní halda a paměť se musí předávat přes třídy jako `Manager`), vlákna paměť sdílejí
* K proměnným mohou přistupovat všechna vlákna programu

### Concurrent design patterns
* Sdílení paměti vyžaduje u vláken obezřetnost při společných modifikacích
* K předcházení chyb vznikly standardizované návrhové vzory
* Lock - zamyká část kódu, aby do ní mohlo vstoupit jen jedno vlákno, metody `lock`/`unlock` nebo `acquire`/`release`
* Read/Write lock - podobný jako Lock, obsahuje oddělený zámek pro čtení a oddělený pro zápis
* Semaphore - definuje se u něj maximální možný počet vláken, povoluje vstup dokud nedojde jeho nastavený limit
* Monitor - Funguje jako synchronizační fronta, využívá metody `wait`, `notify` a `notifyAll`

### Vzor ThreadPool
* Sdružení úloh do fondu vláken
* Obhospodařuje všechny procesy efektivně
* Není potřeba ručně zakládat každé vlákno zvlášť a opakovaně v kódu volat funkce `start()` a `join()` pro jednotlivé instance

### Vzor Producent a Konzument
* Evergreen paralelního programování řešící nevyváženost rychlosti
* Problém se řeší zařazením datové struktury Queue
* K signalizaci, že jsou všechna data předána a vlákno už nemá čekat na další, se do fronty pošle takzvaný End Token (typicky `None`)

### Asynchronní metody (Co-routiny)
* Co-routines jsou asynchronní přístup, kdy po zavolání funkce neproběhne celá najednou
* Zpracuje část operací, svou činnost přeruší slovem yield a navrátí řízení hlavnímu programu
* Opětovné pokračování asynchronní funkce z místa, kde skončila, se vyvolává funkcí `next()`

In [2]:
import threading
from queue import Queue
import time

**1. VLÁKNA A SYNCHRONIZAČNÍ PRIMITIVA (LOCK)**

In [6]:
zamek = threading.Lock()
spolecny_ucet = 0

def vloz_mince(castka, pocet_kusu):
    global spolecny_ucet
    for _ in range(pocet_kusu):
        with zamek:
            spolecny_ucet += castka

t1 = threading.Thread(target=vloz_mince, args=(10, 1000))
t2 = threading.Thread(target=vloz_mince, args=(5, 1000))

t1.start()
t2.start()
t1.join()
t2.join()

print(f"Konečný zůstatek účtu po souběžném běhu vláken: {spolecny_ucet} CZK")

Konečný zůstatek účtu po souběžném běhu vláken: 15000 CZK


**2. NÁVRHOVÝ VZOR PRODUCENT - KONZUMENT**

In [7]:
def producent(fronta):
    print("Producent začal generovat.")
    for pismeno in ["P", "Y", "T", "H", "O", "N"]:
        fronta.put(pismeno)
        time.sleep(0.1)
    fronta.put(None)
    print("Producent skončil.")

def konzument(fronta):
    print("Konzument začal naslouchat.")
    vysledek = []
    while True:
        pismeno = fronta.get()
        if pismeno is None:
            break
        vysledek.append(pismeno)
        time.sleep(0.3)
    print("Konzument poskládal:", "".join(vysledek))

komunikacni_fronta = Queue()

vlakno_producent = threading.Thread(target=producent, args=(komunikacni_fronta,))
vlakno_konzument = threading.Thread(target=konzument, args=(komunikacni_fronta,))

vlakno_producent.start()
vlakno_konzument.start()
vlakno_producent.join()
vlakno_konzument.join()

Producent začal generovat.
Konzument začal naslouchat.
Producent skončil.
Konzument poskládal: PYTHON


**3. ASYNCHRONNÍ METODY (CO-ROUTINES)**

In [8]:
def co_routine():
    yield "První asynchronní část splněna."
    yield "Druhá asynchronní část splněna."

moje_coroutina = co_routine()
print(next(moje_coroutina))
print(next(moje_coroutina))

První asynchronní část splněna.
Druhá asynchronní část splněna.
